In [14]:

"""
============================================================================
TEMEL HFE (HIDDEN FIELD EQUATIONS) KRIPTOSISTEMI - SAGEMATH IMPLEMENTASYONU
============================================================================

Bu modul, yuksek lisans tezinin 5. Bolumunde ("Big Field Tabanli Sistemler")
teorik olarak ele alinan Temel HFE (Hidden Field Equations) kriptosisteminin
SageMath ortaminda somut bir gerceklestirimini sunmaktadir.

TEORIK ARKA PLAN
-----------------
HFE semasi, 1996 yilinda Patarin tarafindan Matsumoto-Imai (MI) kriptosistemine
uygulanan lineerlestirme saldirisina karsi bir cevap olarak onerilmistir. MI
semasindan temel farki, merkez donusumun tek bir Frobenius monomu (X^(q+1))
yerine, derecesi bir D parametresi ile sinirlandirilmis GENEL bir kuadratik
Frobenius polinomu olmasidir:

    F(X) = sum_{0<=i<=j, q^i+q^j<=D} a_ij * X^(q^i+q^j)
         + sum_{0<=i, q^i<=D}        b_i  * X^(q^i)
         + c

Bu yapi, sonlu cisim genislemesi E = F_q[X]/(g(X)) uzerinde tanimlanir ve
X -> X^q Frobenius endomorfizmasinin E uzerinde F_q-lineer olmasi sayesinde,
F(X) fonksiyonu vektor uzayi izomorfizmasi Phi: F_q^n -> E araciligiyla,
F_q^n uzerinde n bilesenli KUADRATIK bir polinom donusumune karsilik gelir.
Bu donusumun kuadratik kalmasinin temel nedeni, E cisminde X^(q^i) ile
X^(q^j) çarpiminin, F_q uzerinde x_1,...,x_n degiskenlerine gore yalnizca
1. dereceden (Frobenius kuvvetleri) terimlerin çarpimindan olustugudur;
dolayisiyla iki lineerin çarpimi olarak ortaya cikan bu terimler F_q
polinom halkasinda ikinci dereceyi asmaz.

Anahtar uretimi, bu "kolay" merkez yapiyi (F) iki rastgele afin (lineer +
oteleme) donusum olan S ve T ile gizleyerek acik anahtari elde eder:

    P = S o Phi^{-1} o F o Phi o T  :  F_q^n -> F_q^n

    Acik Anahtar : P (n adet kuadratik cok degiskenli polinom)
    Gizli Anahtar: S, T (afin donusumler) ve F (merkez HFE polinomu)

Sifreleme, acik anahtar P'nin bir mesaj vektorune uygulanmasindan ibarettir
ve MQ-Problemi (rastgele kuadratik polinom sisteminin cozulmesi) zorlugu
uzerine guvenlik iddiasi kurar. Deşifreleme ise gizli anahtar sayesinde
P'yi katmanlarina ayirarak, zor MQ-Problemini E genisleme cisminde TEK
DEGISKENLI bir polinomun kok bulma problemine indirger; bu da klasik
cisim teorisi araclariyla (Berlekamp, Cantor-Zassenhaus vb. carpanlara
ayirma algoritmalari) polinom zamanda cozulebilir.

Bu implementasyon, tezin 185-195. sayfalarinda anlatilan "Temel HFE
Kriptosistemi" (Bolum 5.2.1) yapisini, her bir ara adimi ekrana basarak
pedagojik/analitik amacli izlenebilir kilacak sekilde tasarlanmistir.

Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
============================================================================
"""

from sage.all import *


class HFE:
    """
    Temel HFE (Hidden Field Equations) kriptosistemini temsil eden sinif.

    Bu sinif, tezin 5.2.1 Bolumunde tanimlanan HFE semasinin anahtar uretimi,
    sifreleme ve desifreleme surecinin tamamini kapsar. Kurulum asamasinda
    (constructor) once temel cisim F_q ve cok degiskenli polinom halkasi
    F_q[x_1,...,x_n] insa edilir; ardindan HFE'nin kalbi olan n. dereceden
    cisim genislemesi E = F_q^n = F_q[X]/(g(X)) olusturulur. Bu genisleme
    cismi, merkez polinom F'nin tanimlandigi ve HFE'nin "gizli kolayligini"
    barindiran uzaydir.

    Onemli Notasyon Esleşmesi (Tez <-> Kod):
        F_q      <-> self.F
        n        <-> self.n (genisleme derecesi / degisken sayisi)
        D        <-> self.D (HFE derece sinirlayici parametresi)
        E        <-> self.E (F_q^n genisleme cismi)
        Phi      <-> vector_to_extension / extension_to_vector metod cifti
        F(X)     <-> self.Secret_Poly (merkez HFE polinomu)
        S, T     <-> self.S_mat/self.S_vec, self.T_mat/self.T_vec (afin donusumler)
        P        <-> self.Public_Key

    Attributes:
        q (int): Temel sonlu cismin eleman sayisi (karakteristik tek olmalidir,
            zira HFE'nin standart tanimi q^i+q^j turu Frobenius toplamlarina
            dayanir; q tek secildiginde carpanlara ayirma asamasinda
            Cantor-Zassenhaus algoritmasinin varsayimlari saglanir).
        n (int): Genisleme cisminin F_q uzerindeki derecesi; ayni zamanda
            hem degisken sayisini hem de vektor uzayi boyutunu belirler.
        D (int): Merkez HFE polinomunun derecesini sinirlayan parametre.
            Tezde vurgulandigi uzere (Bolum 5.2.3.2), D degeri kucuk
            secildiginde HFE polinomunun E uzerindeki matris temsili dusuk
            ranka sahip olur; bu da Rank Saldirisi'nin (Kipnis-Shamir)
            temelini olusturan yapisal zayifliktir. D ayni zamanda
            duzenlilik derecesini (degree of regularity) sinirlayarak
            XL/F4/F5 tipi cebirsel saldirilarin maliyetini de belirler.
        F: F_q uzerindeki temel sonlu cisim (GF(q)).
        R: F_q katsayili, n degiskenli cok degiskenli polinom halkasi
            F_q[x_1,...,x_n]; acik anahtar polinomlari bu halkada yasar.
        vars: R halkasinin uretec degiskenleri (x_1,...,x_n).
        R_uni: F_q katsayili tek degiskenli polinom halkasi F_q[X]; hem
            indirgenemez modulus polinomunun secildigi hem de Phi/Phi^-1
            donusumlerinde ara islemlerin yapildigi halkadir.
        irr_poly: E genisleme cismini insa etmek icin kullanilan, F_q[X]
            icinde n. dereceden indirgenemez polinom (g(X)).
        E: n. dereceden genisleme cismi F_q^n = F_q[X]/(g(X)); HFE'nin
            merkez polinomu F(X) bu cisim uzerinde tanimlidir.
        R_E: E katsayili tek degiskenli polinom halkasi E[X]; merkez
            polinom F(X) bu halkanin bir elemani olarak insa edilir.
    """

    def __init__(self, q, n, D):
        """
        HFE kriptosisteminin temel cebirsel altyapisini kurar.

        Bu asama, anahtar uretiminden BAGIMSIZ olarak, tum sonraki
        islemlerin (anahtar uretimi, sifreleme, desifreleme) uzerine
        insa edilecegi degismez matematiksel yapilari (cisimler, halkalar,
        genisleme) bir kez olusturur.

        Args:
            q (int): Temel sonlu cismin mertebesi (F_q). HFE'nin standart
                tanimi geregi q'nun tek (odd) olmasi onerilir; aksi
                durumda (karakteristik 2), tezde belirtildigi uzere
                (Bolum 5.2.3.2) kuadratik formlarin simetrik matris
                temsili farklilasir ve Rank Saldirisi analizi degisir.
            n (int): Genisleme cisminin derecesi; ayni zamanda hem gizli
                mesaj/sifreli metin vektorlerinin boyutunu hem de acik
                anahtardaki degisken sayisini belirler.
            D (int): Merkez HFE polinomunun derece ust siniri. Guvenlik
                ile verimlilik arasindaki temel odunlesim parametresidir:
                D buyudukce dogrudan saldirilar (MQ-cozumu) zorlasir,
                fakat kok bulma asamasindaki polinom carpanlara ayirma
                maliyeti de artar.

        On Kosullar:
            - q bir asal kuvveti olmalidir (GF(q) fonksiyonunun gecerliligi
              icin SageMath tarafindan kontrol edilir).
            - D >= 2 olmalidir; aksi halde kuadratik terim uretilemez ve
              merkez polinom yalnizca afin bir donusume indirgenir
              (kriptografik olarak anlamsizlasir).

        Yan Etkiler:
            self.F, self.R, self.vars, self.R_uni, self.irr_poly, self.E,
            self.R_E niteliklerini olusturur ve kurulum bilgilerini
            ekrana raporlar.
        """
        self.q = q
        self.n = n
        self.D = D

        print("\n" + "="*80)
        print(f"HFE KRIPTOSISTEMI: TAM DETAYLI ADIM ADIM ANALIZ")
        print("="*80)

        # --------------------------------------------------------------
        # 1. TEMEL CISIM VE COK DEGISKENLI POLINOM HALKASI
        # --------------------------------------------------------------
        # F_q, tum kriptosistemin "yuzeysel" (public-facing) hesaplamalarinin
        # yapildigi zemin cisimdir: acik anahtar P, mesajlar ve sifreli
        # metinler hep bu cisim uzerindeki vektorler/polinomlar olarak ifade
        # edilir. R = F_q[x_1,...,x_n] halkasi, acik anahtarin n adet
        # kuadratik polinomunun tanimlandigi uzaydir.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, n, 'x')
        self.vars = self.R.gens()
        print(f"[KURULUM] Temel Cisim F_q: GF({q})")
        print(f"[KURULUM] Degiskenler: {self.vars}")

        # --------------------------------------------------------------
        # 2. GENISLEME CISMI E = F_q^n
        # --------------------------------------------------------------
        # HFE'nin "gizli kolayligi" tamamen bu genisleme cisminde saklidir:
        # merkez polinom F(X), F_q^n uzerinde karmasik gorunse de, E = F_q[X]/g(X)
        # cisminde TEK DEGISKENLI bir polinom olarak ifade edilir ve bu sayede
        # kok bulma islemleri (deşifreleme) klasik cisim teorisi araclariyla
        # (Berlekamp, Cantor-Zassenhaus vb.) polinom zamanda gerceklestirilebilir.
        # irr_poly, bu genislemeyi tanimlayan n. dereceden indirgenemez
        # polinomdur (g(X)); w ise bu polinomun E icindeki bir kokudur.
        self.R_uni = PolynomialRing(self.F, 'X')
        # Genislemeyi tanimlayacak n. dereceden indirgenemez polinom secilir.
        self.irr_poly = self.R_uni.irreducible_element(n)
        self.E = self.F.extension(self.irr_poly, 'w')

        print(f"[KURULUM] Genisleme Cismi F_(q^n): GF({q}^{n})")
        print(f"[KURULUM] Indirgenemez Polinom (Modulus): {self.irr_poly}")
        print(f"[KURULUM] w, {self.irr_poly} polinomunun GF({q}^{n}) icindeki kokudur.")
        print(f"[KURULUM] w = X mod ({self.irr_poly}) olarak alinir.")
        print("-" * 80)

        # --------------------------------------------------------------
        # 3. E UZERINDE TEK DEGISKENLI POLINOM HALKASI
        # --------------------------------------------------------------
        # Merkez HFE polinomu F(X), bu halkanin (E[X]) bir elemani olarak
        # insa edilecektir; boylece Frobenius monomlari X^(q^i+q^j) dogrudan
        # E katsayili tek degiskenli polinom cebiri araciligiyla ifade edilir.
        self.R_E = PolynomialRing(self.E, 'X')

    # ------------------------------------------------------------------
    # YARDIMCI FONKSIYON: Phi^-1  (E -> F_q^n Vektorlestirme)
    # ------------------------------------------------------------------
    def extension_to_vector(self, element):
        """
        Genisleme cismi E'nin bir elemanini, F_q^n vektor uzayina tasir.

        Bu fonksiyon, tezde Phi^-1 olarak adlandirilan vektor uzayi
        izomorfizmasinin ters yonunu gerceklestirir. E = F_q[X]/(g(X))
        cismindeki bir eleman, {1, w, w^2, ..., w^(n-1)} bazina gore
        acilarak F_q katsayili bir koordinat vektorune donusturulur:

            element = c_0 + c_1*w + c_2*w^2 + ... + c_(n-1)*w^(n-1)
            Phi^-1(element) = (c_0, c_1, ..., c_(n-1)) in F_q^n

        Bu donusum, HFE semasinda hem gizli merkez polinomun ciktisini
        (F(Y) degeri) F_q^n uzayina geri tasimak icin, hem de deşifreleme
        asamasinda bulunan koklerin nihai mesaj adaylarina cevrilmesi icin
        kritik oneme sahiptir.

        Args:
            element: E genisleme cisminin bir elemani (veya E uzerinde
                tanimli bir polinom halkasi elemani, ornegin R_E_multi
                gibi E katsayili cok degiskenli bir halkanin katsayisi).

        Returns:
            vector: F_q uzerinde uzunlugu n olan bir koordinat vektoru.

        Notlar:
            Fonksiyon, gelen 'element' turunun (dogrudan cisim elemani mi,
            yoksa lift/polynomial() metoduna sahip bir sarmalayici mi
            oldugunu) once 'lift()' sonra 'polynomial()' deneyerek tespit
            eder; bu, SageMath'in farkli cisim gosterimleri (finite field
            element vs. quotient ring element) arasindaki uyumu saglar.
        """
        try:
            poly = element.lift()
        except:
            try:
                poly = element.polynomial()
            except:
                poly = element
        poly = self.R_uni(poly)
        coeffs = poly.list()
        # Katsayi listesi n'den kisa gelebilir (yuksek dereceli katsayilar
        # sifir oldugunda SageMath onlari listelemez); eksik konumlar
        # F_q'nun sifir elemani ile doldurulur.
        while len(coeffs) < self.n:
            coeffs.append(self.F(0))
        return vector(self.F, coeffs)

    # ------------------------------------------------------------------
    # YARDIMCI FONKSIYON: Phi  (F_q^n -> E Polinomlastirma)
    # ------------------------------------------------------------------
    def vector_to_extension(self, vec):
        """
        F_q^n uzayindaki bir vektoru, genisleme cismi E'nin bir elemanina tasir.

        Bu fonksiyon, tezde Phi olarak tanimlanan standart vektor uzayi
        izomorfizmasini gerceklestirir:

            Phi(x_1, ..., x_n) = sum_{i=0}^{n-1} x_(i+1) * w^i  in E

        Bu izomorfizma sayesinde, F_q^n uzerindeki gizli/afin islemler
        (T donusumu gibi) E uzerinde tek bir cisim elemanina karsilik
        gelir ve boylece merkez HFE polinomu F(X), bu elemana dogrudan
        uygulanabilir hale gelir.

        Args:
            vec: F_q uzerinde uzunlugu n olan bir vektor (tipik olarak
                gizli T donusumunun ciktisi olan y = T(x) vektoru).

        Returns:
            E cisminin, verilen vektorun w-tabanindaki acilimina karsilik
            gelen elemani.
        """
        w_gen = self.E.gen()
        res = self.E(0)
        for i in range(self.n):
            res += vec[i] * (w_gen**i)
        return res

    # ------------------------------------------------------------------
    # YARDIMCI FONKSIYON: Cisim Denklemleri ile Ideal Indirgeme (x^q - x)
    # ------------------------------------------------------------------
    def reduce_F_ideal(self, polys):
        """
        Verilen polinom vektorunu, F_q'nun "Cisim denklemleri" idealine gore indirger.

        F_q cisminin her elemani s icin s^q = s ozdesligi gecerlidir
        (Fermat'nin kucuk teoreminin genellemesi). Bu nedenle, R = F_q[x_1,...,x_n]
        halkasinda x_i^q - x_i = 0 seklindeki n adet "cisim denklemi" ile
        uretilen ideal, F_q^n uzerinde tanimli fonksiyonlarin polinom
        temsilini KANONIK (en kucuk dereceli) forma indirger.

        Bu islem, HFE anahtar uretiminde merkez polinomun genisleme
        cisminden geri tasinmasi sirasinda ortaya cikan yuksek dereceli
        (fakat F_q^n uzerinde ayni fonksiyonu temsil eden) ifadeleri,
        acik anahtar icin beklenen KUADRATIK forma indirgemek amaciyla
        kullanilir. Indirgeme sonrasinda acik anahtar polinomlarinin
        derecesi 2'yi asmamalidir; aksi durum, teorik yapida bir hata
        oldugunu gosterir.

        Args:
            polys: R halkasinda tanimli polinomlardan olusan bir liste
                veya vektor (indirgenmesi istenen n adet acik anahtar
                bileseni).

        Returns:
            vector: Her biri x_i^q - x_i idealine gore indirgenmis, R
                halkasinda tanimli n adet polinomdan olusan vektor.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)
        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    def generate_keys(self):
        """
        HFE kriptosisteminin acik ve gizli anahtar ciftini uretir.

        Bu metod, tezin 5.2.1 Bolumunde tarif edilen anahtar uretim
        surecinin tam bir uygulamasidir. Sirasiyla:

            1) Gizli afin donusumler S ve T rastgele ve TERSINIR (invertible)
               olacak sekilde uretilir. T "ic katman" (girdi karistirici),
               S ise "dis katman" (cikti karistirici) rolundedir.
            2) Genisleme cismi E uzerinde, derecesi D ile sinirlandirilmis
               rastgele bir HFE merkez polinomu F(X) uretilir. Bu polinom
               hem kuadratik Frobenius terimlerini (X^(q^i+q^j)) hem de
               lineer Frobenius terimlerini (X^(q^i)) icerir.
            3) Acik anahtar, P = S o Phi^-1 o F o Phi o T bileske
               donusumunun SEMBOLIK olarak (degiskenler x_1,...,x_n
               yerine somut sayilar konmadan) hesaplanmasiyla elde edilir.
               Bu hesaplama, HFE'nin "gizli kolayligini" acik anahtarin
               genel gorunumune donusturen kritik adimdir.

        Returns:
            vector: R halkasinda tanimli, n adet kuadratik polinomdan
                olusan acik anahtar P (self.Public_Key olarak da saklanir).

        Yan Etkiler:
            self.T_mat, self.T_vec, self.S_mat, self.S_vec: Gizli afin
                donusumlerin matris/vektor bilesenleri.
            self.Secret_Poly: E[X] halkasinda tanimli gizli merkez HFE
                polinomu F(X).
            self.Public_Key: Uretilen acik anahtar (n polinomdan olusan vektor).
        """
        print("\n" + ">>> BOLUM 1: ANAHTAR URETIMI (KEY GENERATION) <<<")

        # ================================================================
        # ADIM 1: GIZLI AFIN DONUSUMLERIN (S ve T) OLUSTURULMASI
        # ================================================================
        # T ve S, F_q uzerinde tanimli, TERSINIR lineer donusumler (matris)
        # ile bir oteleme vektorunun birlesiminden olusan afin donusumlerdir:
        #   T(x) = M_T * x + c_T ,   S(x) = M_S * x + c_S
        # Tersinirlik sarti zorunludur; cunku desifreleme asamasinda bu
        # donusumlerin TERSI (M_T^-1, M_S^-1) kullanilarak orijinal mesaja
        # geri donulur. Bu iki katman, merkez polinomun F_q^n uzerindeki
        # kolayca cozulebilir cebirsel yapisini (dusuk rank, ozel Frobenius
        # formu) acik anahtarda gizlemekle gorevlidir.
        print("\n[ADIM 1] Gizli Afin Donusumler (S ve T) Uretiliyor...")
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible():
                break
        self.T_vec = random_vector(self.F, self.n)

        while True:
            self.S_mat = random_matrix(self.F, self.n, self.n)
            if self.S_mat.is_invertible():
                break
        self.S_vec = random_vector(self.F, self.n)

        print("   -> T Donusumu (Ic Katman - x'i karistirir):")
        print(f"      Matris (M_T):\n{self.T_mat}")
        print(f"      Vektor (c_T): {self.T_vec}")
        print("   -> S Donusumu (Dis Katman - Ciktiyi karistirir):")
        print(f"      Matris (M_S):\n{self.S_mat}")
        print(f"      Vektor (c_S): {self.S_vec}")

        # ================================================================
        # ADIM 2: GIZLI HFE MERKEZ POLINOMU F(X)'IN URETILMESI
        # ================================================================
        # Tezde tanimlanan genel HFE merkez polinomu su forma sahiptir:
        #
        #   F(X) = sum_{0<=i<=j, q^i+q^j<=D} a_ij * X^(q^i+q^j)      (kuadratik terimler)
        #        + sum_{0<=i, q^i<=D}        b_i  * X^(q^i)          (lineer Frobenius terimleri)
        #        + c                                                  (sabit terim)
        #
        # Bu polinomun E[X] halkasinda TEK DEGISKENLI olarak insa edilmesi,
        # HFE'nin temel "numarasidir": F_q^n uzerinde kuadratik gorunen bir
        # donusum, E uzerinde tek degiskenli, dusuk terimli bir polinoma
        # indirgenir ve bu sayede kokleri polinom zamanda bulunabilir hale
        # gelir. D parametresi, kuadratik terim sayisini sinirlayarak hem
        # kok bulma maliyetini hem de (tez Bolum 5.2.3.2'de aciklanan)
        # Rank Saldirisi'na karsi guvenlik/verimlilik dengesini belirler.
        print("\n[ADIM 2] Gizli HFE Polinomu F(X) Uretiliyor...")
        X_uni = self.R_E.gen()
        P = self.R_E(0)

        # --- Kuadratik HFE terimleri: sum a_ij * X^(q^i + q^j) ---
        # Yalnizca q^i + q^j <= D kosulunu saglayan Frobenius kuvvet
        # ciftleri (i,j) icin rastgele katsayilar (a_ij in E) atanir.
        # i>=j sinirlamasi (j'nin 0'dan i+1'e kadar dolasmasi), X^(q^i+q^j)
        # ile X^(q^j+q^i)'nin ayni monom oldugu icin cift sayimi engeller.
        terms_log = []

        for i in range(self.n):
            for j in range(i + 1):
                deg = self.q**i + self.q**j
                if deg <= self.D:
                    coeff = self.E.random_element()
                    if coeff != 0:
                        P += coeff * (X_uni ** deg)
                        terms_log.append(f"{coeff}*X^{deg}")

        # --- Lineer Frobenius terimleri: sum b_i * X^(q^i) ---
        # Bu terimler F_q^n uzerinde acildiginda yine LINEER kalirlar
        # (Frobenius endomorfizmasinin F_q-lineerligi geregi); dolayisiyla
        # nihai polinomun kuadratik derecesini etkilemezler, fakat merkez
        # donusumu zenginlestirerek saldirilara karsi ek karmasiklik katarlar.
        linear_terms_log = []

        for i in range(self.n):
            deg = self.q**i
            coeff = self.E.random_element()
            if coeff != 0:
                P += coeff * (X_uni ** deg)
                linear_terms_log.append(f"{coeff}*X^{deg}")

        # --- Sabit terim: c ---
        # F_q^n uzerinde acildiginda sabit (0. dereceden) bir vektore
        # karsilik gelir; donusumun afinligini pekistirir.
        const_coeff = self.E.random_element()
        P += const_coeff

        self.Secret_Poly = P

        print(f"   -> Olusturulan F(X) Polinomu (Extension Field uzerinde):")
        print(f"      Derece: {P.degree()}")
        print(f"      Kuadratik HFE terimleri: {terms_log}")
        print(f"      Lineer Frobenius terimleri: {linear_terms_log}")
        print(f"      Sabit terim: {const_coeff}")
        print(f"      F(X) = {P}")
        # ================================================================
        # ADIM 3: ACIK ANAHTAR HESAPLAMASI
        #         E(x) = S( Phi^-1( F( Phi( T(x) ) ) ) )
        # ================================================================
        # Bu asama, HFE anahtar uretiminin kalbidir: gizli katmanlarin
        # (T, F, S) BILESKESI sembolik olarak (x_1,...,x_n degiskenlerini
        # somut degerlere indirgemeden) hesaplanarak, sonucun F_q[x_1,...,x_n]
        # halkasinda KUADRATIK bir polinom sistemine indirgendigi dogrudan
        # gozlemlenir. Islem, izlenebilirlik amaciyla asamalara (A-F)
        # bolunmustur.
        print("\n[ADIM 3] Acik Anahtar Hesabi: E(x) = S( Phi^-1( F( Phi( T(x) ) ) ) )")
        print("   Bu kisim sembolik olarak hesaplanacak ve ara adimlar gosterilecek.")

        # --- A) T(x): Ic katmanin sembolik uygulanmasi ---
        # x = (x_1,...,x_n) degisken vektorune T afin donusumu uygulanir.
        # Sonuc, hala R halkasinda (F_q[x_1,...,x_n]) tanimli, LINEER
        # bilesenlerden olusan bir vektordur.
        x_vec = vector(self.R, self.vars)
        y_vec = self.T_mat * x_vec + self.T_vec
        print(f"\n   A) T(x) Sembolik Ciktisi (Vektor):")
        print(f"      {y_vec}")

        # --- B) Phi(T(x)) -> Y: Genisleme cismine sembolik tasima ---
        # y_vec'in her bir bileseni, KATSAYILARI E cisminde olan cok
        # degiskenli bir halkaya (R_E_multi) tasinir; ardindan Phi
        # izomorfizmasi (w-tabaninda agirlikli toplam) sembolik olarak
        # uygulanarak, x_1,...,x_n degiskenlerine bagli TEK BIR E-degerli
        # polinom Y elde edilir. Bu adim, F_q^n uzerindeki T(x) vektorunu,
        # merkez polinom F'nin girdi uzayi olan E'ye tasimanin sembolik
        # karsiligidir.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])
        w_gen = self.E.gen()
        Y = sum(y_vec_E[i] * (w_gen**i) for i in range(self.n))

        print(f"\n   B) Phi(T(x)) -> Y (Extension Field Elemani olarak temsil):")
        print(f"      Y ifadesi cok uzundur, ancak x degiskenlerine bagli bir polinomdur.")
        print(f"      Y (sembolik): {Y}")

        # --- C) Z = F(Y): Merkez donusumun sembolik uygulanmasi ---
        # Gizli merkez polinom P(X) = F(X), Y ifadesine (yani x_1,...,x_n'e
        # bagli sembolik girdiye) uygulanir. Bu, HFE'nin "gizli/hidden"
        # yapisini olusturan asamadir: P'nin E uzerindeki basit Frobenius
        # yapisi, x_1,...,x_n degiskenlerine bagli COK YUKSEK dereceli
        # (henuz indirgenmemis) bir polinoma donusur.
        print(f"\n   C) F(Y) Hesaplaniyor... ('Hidden' yapiyi olusturur)")
        Z = R_E_multi(0)
        p_dict = P.dict()

        for deg, coeff in p_dict.items():
            Z += coeff * (Y**deg)

        print(f"      -> Ham Z Polinomu Derecesi: {Z.degree()} (Indirgenmeden once cok yuksek!)")

        # --- D) Cisim denklemleriyle indirgeme (x_i^q = x_i) ---
        # Z polinomu, F_q^n uzerinde bir fonksiyon olarak Ham (yuksek
        # dereceli) gorunse de, x_i^q - x_i = 0 cisim denklemleri idealine
        # gore indirgendiginde HFE teorisinin ongordugu gibi KUADRATIK
        # dereceye duser. Bu, HFE'nin merkezi teorik iddiasinin (Frobenius
        # kuvvetlerinin F_q uzerinde lineer davranmasi) somut dogrulamasidir.
        vars_E = R_E_multi.gens()
        field_eqs_E = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs_E)
        Z_reduced = Ideal_E.reduce(Z)

        print(f"      -> Cisim denklemleriyle indirgendi (x^q = x).")
        print(f"      -> Yeni Derece: {Z_reduced.degree()} (Burasi en fazla 2 olmali!)")

        # --- E) Phi^-1(Z): E-katsayili polinomdan F_q-katsayili vektore donus ---
        # Z_reduced polinomunun her monomunun E-degerli katsayisi,
        # extension_to_vector (Phi^-1) araciligiyla F_q^n'e tasinir ve
        # ilgili monomiyle carpilarak n adet ayri F_q[x_1,...,x_n]
        # polinomunun (z_polys) ilgili bilesenine eklenir. Sonuc, S
        # katmani uygulanmadan hemen onceki "ara" merkez donusumun
        # (Phi^-1 o F o Phi o T) acik ifadesidir.
        z_polys = [self.R(0) for _ in range(self.n)]
        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff)
            monomial = self.R.monomial(*exps)
            for i in range(self.n):
                z_polys[i] += coeff_vec[i] * monomial
        z_vec = vector(self.R, z_polys)

        print(f"\n   D) Z ve Phi^-1(Z) -> Ara Vektor (S uygulanmadan onceki hali):")
        print(f"      {z_polys}")
        print(f"      {z_vec}")

        # --- F) S(z): Dis katmanin uygulanmasi ve cisim idealiyle nihai indirgeme ---
        # z_vec vektorune S afin donusumu uygulanarak nihai acik anahtar
        # elde edilir. Son bir kez daha cisim idealine gore indirgenmesi
        # (reduce_F_ideal), S donusumunun matris carpiminin da x_i^q = x_i
        # ozdesligiyle tutarli, kanonik (minimal dereceli) bir forma
        # getirilmesini saglar.
        raw_pk = self.S_mat * z_vec + self.S_vec
        self.Public_Key = self.reduce_F_ideal(raw_pk)

        print(f"\n   E) S(Ara Vektor) -> Acik Anahtar (Public Key):")
        print(f"      Elde edilen polinom seti (P_1, ..., P_n):")
        for i, p in enumerate(self.Public_Key):
            print(f"      P_{i} = {p}")

        return self.Public_Key

    def encrypt(self, message_vec):
        """
        Verilen bir duz metin (plaintext) vektorunu acik anahtar ile sifreler.

        Tezde tanimlanan HFE sifreleme islemi son derece basittir: mesaj
        vektoru z, acik anahtar P'nin n adet kuadratik polinomunda
        DEGISKENLERIN YERINE KONMASI (substitution) yoluyla degerlendirilir:

            w = P(z) = (P_1(z), P_2(z), ..., P_n(z))  in F_q^n

        Bu islemin tek yonlu (kolay hesaplanabilir, ancak tersinin zor
        olmasi beklenen) dogasi, MQ-Probleminin (rastgele kuadratik cok
        degiskenli polinom sisteminin cozulmesi) NP-Zor oldugu varsayimina
        dayanir; alicinin gizli anahtari bilmedikce bu tersine cevirmeyi
        verimli sekilde gerceklestiremeyecegi varsayilir.

        Args:
            message_vec: F_q uzerinde uzunlugu n olan duz metin vektoru
                (sifrelenecek mesaj z).

        Returns:
            vector: F_q uzerinde uzunlugu n olan sifreli metin vektoru
                (c = P(z)).
        """
        print("\n" + ">>> BOLUM 2: SIFRELEME (ENCRYPTION) <<<")
        print(f"   Mesaj (m): {message_vec}")
        val_dict = {self.vars[i]: message_vec[i] for i in range(self.n)}

        print("   Acik anahtar polinomlarinda yerine koyuluyor...")
        ciphertext = []
        for i, p in enumerate(self.Public_Key):
            val = p.subs(val_dict)
            ciphertext.append(val)
            # print(f"      P_{i}(m) = {val}") # Cok yer kaplarsa kapatilabilir

        res = vector(self.F, ciphertext)
        print(f"   Sifreli Metin (c): {res}")
        return res

    def decrypt(self, ciphertext_vec):
        """
        Verilen bir sifreli metni, gizli anahtar bilgisiyle desifre eder.

        Bu metod, tezde detaylandirilan "HFE'nin desifreleme prensibini"
        (gizli anahtar sayesinde MQ-Probleminin, genisleme cismi E uzerinde
        TEK DEGISKENLI bir kok bulma problemine indirgenmesi) adim adim
        uygular:

            1) Dis katman S'in tersi alinarak sifreli metinden S'in etkisi
               kaldirilir: w = S^-1(y - c_S).
            2) w vektoru, Phi izomorfizmasi ile genisleme cismi E'ye
               tasinir: W = Phi(w).
            3) Gizli merkez polinom F(X) = P(X) kullanilarak F(X) - W = 0
               denkleminin E icindeki TUM kokleri bulunur. Bu adim,
               HFE'nin guvenligini/verimliligini belirleyen kritik
               noktadir: E uzerinde tek degiskenli bir polinomun kokleri,
               Berlekamp veya Cantor-Zassenhaus gibi klasik carpanlara
               ayirma algoritmalariyla POLINOM ZAMANDA bulunabilir
               (bkz. tez Bolum 4.1), oysa ayni problem F_q^n uzerinde
               dogrudan (MQ-Problemi olarak) COZULEMEZ zorlukta kabul edilir.
            4) Bulunan her kok, Phi^-1 ile F_q^n'e geri tasinir.
            5) Ic katman T'nin tersi uygulanarak orijinal mesaj adaylarina
               ulasilir: x = T^-1(y' - c_T).

        Onemli Not: HFE polinomunun kuadratik yapisi geregi F(X) - W = 0
        denklemi genel olarak BIRDEN FAZLA koke sahip olabilir; bu nedenle
        desifreleme sureci potansiyel olarak birden fazla ADAY mesaj
        uretebilir. Gercek uygulamalarda dogru adayi belirlemek icin
        (orn. bir hash/redundancy kontrolu ile) ek bir mekanizmaya
        ihtiyac duyulur; bu ornek implementasyonda tum adaylar dondurulup
        dogrulama, cagiran kod tarafinda yapilmaktadir.

        Args:
            ciphertext_vec: F_q uzerinde uzunlugu n olan sifreli metin
                vektoru y.

        Returns:
            list: F_q uzerinde uzunlugu n olan, F(X) - W = 0 denkleminin
                her bir kokune karsilik gelen aday duz metin vektorlerinin
                listesi.
        """
        print("\n" + ">>> BOLUM 3: DESIFRELEME (DECRYPTION) <<<")
        print(f"   Sifreli Metin (y): {ciphertext_vec}")

        # ================================================================
        # ADIM 1: DIS KATMANIN (S) TERSINE CEVRILMESI
        # ================================================================
        # S afin donusumunun tersi alinarak, sifreli metinden S'in
        # eklendigi "karistirma" etkisi kaldirilir: w = S^-1 * (y - c_S).
        print("\n   [ADIM 1] Dis Katman (S) Tersine Cevriliyor...")
        S_inv = self.S_mat.inverse()
        print(f"      S^-1 Matrisi:\n{S_inv}")
        w_vec = S_inv * (ciphertext_vec - self.S_vec)
        print(f"      Hesap: S^-1 * (y - c_S)")
        print(f"      Sonuc (w): {w_vec}")

        # ================================================================
        # ADIM 2: VEKTORDEN GENISLEME CISMI ELEMANINA GECIS (Phi)
        # ================================================================
        # w vektoru, standart vektor uzayi izomorfizmasi Phi araciligiyla
        # E genisleme cismindeki tekabul eden elemana (W) donusturulur.
        # Bu adim, F_q^n uzerinde MQ-Problemi olarak gorunen zorlugu, E
        # uzerinde tek degiskenli bir denkleme indirgemenin ilk kosuludur.
        print("\n   [ADIM 2] Vektor -> Genisleme Cismi Elemani (Phi)...")
        W = self.vector_to_extension(w_vec)
        print(f"      w vektoru Genisleme cisminde suna denk gelir:")
        print(f"      W = {W}")

        # ================================================================
        # ADIM 3: GIZLI POLINOMUN KOKLERININ BULUNMASI
        # ================================================================
        # Merkez HFE polinomu ile W arasindaki fark alinarak, P(X) - W = 0
        # denklemi kurulur. Bu denklemin E icindeki kokleri, F(X) merkez
        # donusumu altinda W'ye eslenen tum X degerlerini (yani F^-1(W)
        # kumesini) verir. Kok bulma islemi, arka planda SageMath'in cisim
        # genislemeleri uzerindeki carpanlara ayirma rutinlerini (kavramsal
        # olarak Berlekamp/Cantor-Zassenhaus ailesine dayanan algoritmalari,
        # bkz. tez Bolum 4.1) kullanir.
        print("\n   [ADIM 3] Gizli Polinomun Koklerini Bulma...")
        print(f"      Denklem: F(X) = W  =>  F(X) - W = 0")
        print(f"      F(X) - W = {self.Secret_Poly - W}")

        roots = (self.Secret_Poly - W).roots()
        print(f"      -> Bulunan Kok Sayisi: {len(roots)}")

        candidates = []
        T_inv = self.T_mat.inverse()

        print("\n   [ADIM 4 & 5] Kokleri Vektore Cevirme ve T'yi Tersine Cevirme")
        print(f"      T^-1 Matrisi:\n{T_inv}")

        # ================================================================
        # ADIM 4 & 5: HER KOK ICIN PHI^-1 VE T^-1 UYGULANMASI
        # ================================================================
        # Bulunan her kok r, potansiyel bir orijinal Y = Phi(T(x)) degerine
        # karsilik gelir. Bu nedenle her kok icin sirasiyla:
        #   (a) Phi^-1 uygulanarak F_q^n uzayina donus yapilir (y'),
        #   (b) T'nin tersi uygulanarak gizli ic katman kaldirilir ve
        #       nihai aday duz metin x = T^-1 * (y' - c_T) elde edilir.
        # Birden fazla kok bulunmasi durumunda, bu birden fazla ADAY
        # mesaj anlamina gelir; dogru aday harici bir dogrulama
        # mekanizmasiyla (bu ornekte cagiran kod tarafinda) belirlenir.
        for idx, (r, mult) in enumerate(roots):
            print(f"\n      --- ADAY KOK {idx+1} ---")
            print(f"      a) Kok (R) [Extension]: {r}")

            # (a) Phi^-1: Genisleme cismi elemanindan F_q^n vektorune donus.
            y_vec = self.extension_to_vector(r)
            print(f"      b) Vektor (y') [Phi^-1(R)]: {y_vec}")

            # (b) T^-1: Ic katmanin tersinin uygulanmasi.
            plaintext = T_inv * (y_vec - self.T_vec)
            print(f"      c) T^-1 Uygulamasi (x = T^-1 * (y' - c_T)):")
            print(f"         ADAY MESAJ: {plaintext}")

            candidates.append(plaintext)

        return candidates




In [15]:
# ============================================================================
# SENARYO CALISTIRMA
# ============================================================================
# Asagidaki blok, yukarida tanimlanan HFE sinifinin uctan uca (anahtar
# uretimi -> sifreleme -> desifreleme -> dogrulama) calistirilmasini
# gosteren somut bir ornek teskil eder. Parametreler, hesaplama ciktilarinin
# ekranda okunabilir kalmasi amaciyla kucuk tutulmustur; gercek kriptografik
# guvenlik icin cok daha buyuk q, n ve D degerleri gereklidir.
# ============================================================================

# Parametreler: Okunabilirlik icin kucuk cisim seciyoruz
my_q = 7   # Cisim boyutu (F_q'nun eleman sayisi)
my_n = 3   # Degisken sayisi / genisleme derecesi (vektor boyutu)
my_D = 50  # Derece parametresi (HFE merkez polinomunun derece sinirlayicisi)

hfe = HFE(q=my_q, n=my_n, D=my_D)

# 1. Anahtar Uretimi: Gizli (S, F, T) ve acik (P) anahtarlarin olusturulmasi
pk = hfe.generate_keys()

# 2. Test Verisi: F_q^n uzayindan rastgele bir duz metin (plaintext) secimi
msg = random_vector(GF(my_q), my_n)

# 3. Sifreleme: Acik anahtar kullanilarak mesajin sifrelenmesi
cipher = hfe.encrypt(msg)

# 4. Desifreleme: Gizli anahtar kullanilarak sifreli metnin cozulmesi
candidates = hfe.decrypt(cipher)

# 5. Sonuc Kontrolu: Desifreleme sonucunda elde edilen aday mesajlar
#    arasinda orijinal mesajin bulunup bulunmadiginin dogrulanmasi.
print("\n" + "="*60)
print("SONUC DEGERLENDIRMESI")
print("="*60)
print(f"Orijinal Mesaj: {msg}")
print(f"Bulunan Adaylar: {len(candidates)}")

found = False
for i, cand in enumerate(candidates):
    if cand == msg:
        print(f"BASARILI! Aday {i+1} orijinal mesajla birebir eslesiyor.")
        found = True

if not found:
    print("HATA: Orijinal mesaj adaylar arasinda bulunamadi.")



HFE KRIPTOSISTEMI: TAM DETAYLI ADIM ADIM ANALIZ
[KURULUM] Temel Cisim F_q: GF(7)
[KURULUM] Degiskenler: (x0, x1, x2)
[KURULUM] Genisleme Cismi F_(q^n): GF(7^3)
[KURULUM] Indirgenemez Polinom (Modulus): X^3 + 6*X^2 + 4
[KURULUM] w, X^3 + 6*X^2 + 4 polinomunun GF(7^3) icindeki kokudur.
[KURULUM] w = X mod (X^3 + 6*X^2 + 4) olarak alinir.
--------------------------------------------------------------------------------

>>> BOLUM 1: ANAHTAR URETIMI (KEY GENERATION) <<<

[ADIM 1] Gizli Afin Donusumler (S ve T) Uretiliyor...
   -> T Donusumu (Ic Katman - x'i karistirir):
      Matris (M_T):
[2 2 0]
[2 1 6]
[6 2 2]
      Vektor (c_T): (0, 4, 5)
   -> S Donusumu (Dis Katman - Ciktiyi karistirir):
      Matris (M_S):
[0 5 0]
[5 3 4]
[1 4 4]
      Vektor (c_S): (6, 6, 3)

[ADIM 2] Gizli HFE Polinomu F(X) Uretiliyor...
   -> Olusturulan F(X) Polinomu (Extension Field uzerinde):
      Derece: 50
      Kuadratik HFE terimleri: ['4*w^2 + 3*w*X^2', 'w^2 + 5*w + 2*X^8', '5*w^2 + 4*w + 3*X^14', '3*w +